# 第 1 章习题与解答

> 本章是「大图景」,习题偏向理解而非编码。后续章节会有更多动手题。

## Exercise 1.1

**题目**:minimind 的词表大小是 6400,而 GPT-4 约 10 万。请思考:如果把 minimind 的词表扩大到 10 万,会带来什么好处和坏处?对 64M 参数的模型来说合理吗?

<details><summary><b>参考答案</b></summary>

**好处**:
- 每个中文/英文词更可能作为一个整体 token 出现(而非被拆成多个子词),减少序列长度,推理更快。
- 对罕见词的表示更准确。

**坏处**:
- `embed_tokens` 层参数 = vocab_size × hidden_size。从 6400×768(490 万)涨到 100000×768(7680 万)—— **词表层就占了模型总参数的 12 倍以上**,严重失衡。
- 绑定权重(`tie_word_embeddings=True`)意味着 lm_head 也跟着膨胀。
- 训练数据不足以让 10 万个 token 都得到充分训练,大部分 token 的 embedding 会接近随机。

**结论**:对于 64M 的小模型,6400 词表是合理的工程取舍 —— 在参数预算有限时,把参数花在 Transformer 层(真正学习语言规律的部位)比花在词表上更划算。

</details>

## Exercise 1.2

**题目**:在 `eval_llm.py:73-74` 中有这样一段:

```python
if 'pretrain' in args.weight:
    inputs = tokenizer.bos_token + prompt
else:
    inputs = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
```

为什么预训练权重和 SFT 权重要用不同的输入格式?如果用反了会怎样?

<details><summary><b>参考答案</b></summary>

**原因**:两种权重训练时见过的「输入格式」不同。

- **预训练阶段**(`train_pretrain.py`):模型只在**纯文本**上训练(`PretrainDataset` 返回 `[bos] + tokens + [eos]`),从未见过 `<|im_start|>` 这类对话模板 token。如果给它喂对话模板,它会产生「陌生 token → 随机 logits → 胡言乱语」。
- **SFT 阶段**(`train_full_sft.py`):模型在 `apply_chat_template` 格式的对话数据上训练,学会了「看到 `<|im_start|>assistant\n` 就开始续写 assistant 的回复」。如果给它喂纯文本,它不知道该扮演什么角色。

**用反了的后果**:
- SFT 权重 + 纯文本输入 → 模型可能续写文本而不是回答问题(它在 SFT 时学的是「对话续写」)。
- 预训练权重 + 对话模板输入 → 模型遇到没见过的特殊 token,输出不可预测的乱码。

这个细节揭示了 LLM 的一个核心事实:**模型的输入格式必须和训练时的格式一致**。这也是为什么 SFT 数据集的构造(第 9 章)如此重要。

</details>

## Exercise 1.3

**题目**:minimind 的 `generate` 方法(第 7 章详解)默认参数是 `temperature=0.85, top_p=0.95`。请尝试回答:

1. 如果把 `temperature` 设为 `0`(或极小值),生成会变成什么样?
2. 如果把 `top_p` 设为 `1.0`,会发生什么?

(提示:回想 1.4.3 节关于采样的说明。第 7 章会有完整推导。)

<details><summary><b>参考答案</b></summary>

1. **temperature → 0**:softmax 后概率分布变得极端(指数除以接近 0 的值会放大差异),几乎等价于 **greedy decoding**(每步选 logits 最大的 token)。生成结果**完全确定** —— 同样的输入永远产生同样的输出。好处是可复现,坏处是容易陷入重复循环(所以需要 repetition_penalty)。

2. **top_p = 1.0**:不丢弃任何 token,在**整个词表**上采样。这意味着极低概率的 token(模型几乎不认可的词)也有机会被选中,生成可能出现「跳跃性」的奇怪输出。实践中 `top_p=0.9~0.95` 是常见选择,在多样性和质量之间取平衡。

> **延伸思考**:为什么 minimind 不用 `top_k`?在 6400 的小词表上,`top_k=50` 已经覆盖了词表的 0.78%,可能过于严格。`top_p` 自适应概率分布,更稳健。(第 7 章会对比两种策略。)

</details>